# E.R.M.E.S. - Fase 4: Progettazione e Addestramento della CNN Custom

In questo notebook abbandoniamo le *engineered features* del Machine Learning classico (Flattening + PCA) per passare alle **learned features** del Deep Learning. Costruiamo l'architettura fondamentale del progetto, una Rete Neurale Convoluzionale (CNN) progettata per estrarre la gerarchia spaziale dai volti.

Implementeremo una pipeline completa:
1. **Data Engineering (Pipeline Asincrona):** Ottimizzazione I/O con `tf.data`, separando il carico della `Data Augmentation` sulla CPU per non generare colli di bottiglia sulla GPU.
2. **Architettura VGG-like:** Blocchi sequenziali di Conv2D per l'estrazione gerarchica dei pattern visivi (bordi $\rightarrow$ texture $\rightarrow$ micro-espressioni).
3. **Regolarizzazione:** Uso massiccio di `BatchNormalization` (per stabilizzare i gradienti) e `Dropout` (per forzare reti robuste e combattere l'overfitting).
4. **Controllo Dinamico (AutoML base):** Implementazione di `EarlyStopping`, `ReduceLROnPlateau` e `ModelCheckpoint`.
5. **Class Imbalance:** Iniezione diretta dei `class_weights` nella funzione di costo (Cost-sensitive learning).

In [ ]:
"""
Inizializzazione dell'ambiente e ricostruzione della pipeline dati ottimizzata.
L'uso di AUTOTUNE permette a TensorFlow di allocare dinamicamente i thread della CPU 
per preparare i batch di immagini (Augmentation) mentre la GPU calcola i gradienti.
"""
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path
import logging

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"GPU rilevata e configurata per il Memory Growth: {len(gpus)} GPU(s)")
    except RuntimeError as e:
        print(e)

tf.get_logger().setLevel(logging.ERROR)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_context("notebook", font_scale=1.1)

BATCH_SIZE = 64
IMG_SIZE = (48, 48)
AUTOTUNE = tf.data.AUTOTUNE

print("--- COSTRUZIONE PIPELINE DATI ---")

# Blocco di Augmentation Spaziale (Mitiga il rumore e i volti disallineati)
data_augmentation = tf.keras.Sequential([
    layers.Rescaling(1./255),
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.1, 0.1)
])

# Caricamento Grezzo
train_ds_raw = tf.keras.utils.image_dataset_from_directory(
    '../data/fer2013/train', color_mode='grayscale', image_size=IMG_SIZE, 
    batch_size=BATCH_SIZE, shuffle=True, label_mode='int'
)

val_ds_raw = tf.keras.utils.image_dataset_from_directory(
    '../data/fer2013/test', color_mode='grayscale', image_size=IMG_SIZE, 
    batch_size=BATCH_SIZE, shuffle=False, label_mode='int'
)

# Pipeline
train_ds = train_ds_raw.cache().map(
    lambda x, y: (data_augmentation(x, training=True), y),
    num_parallel_calls=AUTOTUNE
).prefetch(buffer_size=AUTOTUNE)

val_ds = val_ds_raw.cache().map(
    lambda x, y: (x / 255.0, y),
    num_parallel_calls=AUTOTUNE
).prefetch(buffer_size=AUTOTUNE)

# Pesi calcolati nell'EDA per il Cost-sensitive learning
class_weights = {
    0: 1.0266, 1: 9.4066, 2: 1.0010, 3: 0.5684, 
    4: 0.8260, 5: 0.8491, 6: 1.2934
}

print("Pipeline asincrona per GPU configurata con successo")

In [ ]:
def build_ermes_cnn() -> models.Sequential:
    """
    Costruisce l'architettura della CNN. 
    Struttura: Feature Extractor (Conv2D) + Classificatore (Dense).
    """
    model = models.Sequential(name="ERMES_CNN_Custom")
    
    # --- FEATURE EXTRACTOR ---
    model.add(layers.InputLayer(input_shape=(48, 48, 1)))
    
    # Blocco 1: Basso Livello (Cerca linee, bordi, contorni del viso)
    model.add(layers.Conv2D(64, (3, 3), padding='same', activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.Conv2D(64, (3, 3), padding='same', activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D(pool_size=(2, 2)))
    model.add(layers.Dropout(0.25))
    
    # Blocco 2: Medio Livello (Cerca texture, occhi, narici, denti)
    model.add(layers.Conv2D(128, (3, 3), padding='same', activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.Conv2D(128, (3, 3), padding='same', activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D(pool_size=(2, 2)))
    model.add(layers.Dropout(0.35))
    
    # Blocco 3: Alto Livello (Assembla i tratti in micro-espressioni emotive)
    model.add(layers.Conv2D(256, (3, 3), padding='same', activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.Conv2D(256, (3, 3), padding='same', activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D(pool_size=(2, 2)))
    model.add(layers.Dropout(0.45))
    
    # --- CLASSIFICATORE LINEARE ---
    model.add(layers.Flatten())
    # Strato denso per l'interpretazione delle feature astratte
    model.add(layers.Dense(512, activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.Dropout(0.5)) # Alta penalità per impedire la memorizzazione
    
    # Output: 7 neuroni con attivazione Softmax per estrarre probabilità percentuali
    model.add(layers.Dense(7, activation='softmax')) 
    
    # Compilazione
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

cnn_model = build_ermes_cnn()
cnn_model.summary()

In [ ]:
"""
Configurazione del controllo dinamico del training (AutoML base) 
e sistema di Experiment Tracking (TensorBoard).
"""
import datetime
import os

# Creazione dinamica della cartella di log basata sul timestamp esatto
log_dir = os.path.join("logs", "fit", datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))

callbacks_list = [
    # 1. Early Stopping: Ferma il training per evitare l'overfitting
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=15,          
        restore_best_weights=True,
        verbose=1
    ),
    
    # 2. Reduce LR: Adatta il learning rate se la loss smette di scendere
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,           
        patience=5,           
        min_lr=1e-6,
        verbose=1
    ),
    
    # 3. Model Checkpoint: Salva l'artefatto fisico su disco
    tf.keras.callbacks.ModelCheckpoint(
        filepath='ermes_best_cnn.h5',
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    ),
    
    # 4. Experiment Tracking: Salva le metriche, i grafici architetturali e la distribuzione dei pesi
    tf.keras.callbacks.TensorBoard(
        log_dir=log_dir, 
        histogram_freq=0,
        write_graph=True
    )
]

print(f"Callbacks configurate. I log di TensorBoard verranno salvati in: {log_dir}")

In [ ]:
"""
Esecuzione del Training con iniezione del Class Weighting per 
bilanciare l'importanza della Loss Function.
"""
EPOCHS = 100

print("Inizio Addestramento CNN E.R.M.E.S. ...\n")
history = cnn_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    class_weight=class_weights,
    callbacks=callbacks_list
)

In [ ]:
"""
Avvio dell'interfaccia interattiva TensorBoard.
Permette di esplorare visivamente lo storico degli esperimenti, 
sovrapporre i run e ispezionare l'architettura della rete (Graphs).
"""
# Carica l'estensione di TensorBoard per Jupyter
%load_ext tensorboard

# Lancia la dashboard puntando alla cartella root dei log
%tensorboard --logdir logs/fit

In [ ]:
import tensorflow as tf

# --- CARICAMENTO DEL MODELLO ---
try:
    cnn_model = tf.keras.models.load_model('ermes_best_cnn.h5')
    print("Modello NUOVO caricato con successo dalla cartella corrente")
except:
    cnn_model = tf.keras.models.load_model('../models/ermes_best_cnn.h5')
    print("Modello PRE-ESISTENTE caricato con successo dalla cartella 'models'")

cnn_model.summary()

In [ ]:
import os
from IPython.display import SVG, display

def plot_training_history(history):
    """
    Renderizza i grafici dell'accuratezza e della loss per verificare visivamente
    l'assenza di overfitting e la convergenza del modello.
    """
    acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    epochs_range = range(1, len(acc) + 1)
    
    # Troviamo l'epoca migliore (dove val_loss è minima)
    best_epoch = np.argmin(val_loss) + 1

    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    fig.suptitle('Analisi della Convergenza - Modello E.R.M.E.S. Custom', fontsize=16, fontweight='bold')
    
    # 1. Grafico Accuracy
    axes[0].plot(epochs_range, acc, label='Training Accuracy', color='#2ca02c', linewidth=2.5)
    axes[0].plot(epochs_range, val_acc, label='Validation Accuracy', color='#1f77b4', linewidth=2.5)
    axes[0].axvline(x=best_epoch, color='red', linestyle='--', alpha=0.7, label=f'Best Epoch ({best_epoch})')
    axes[0].set_title('Progressione dell\'Accuratezza', fontsize=14)
    axes[0].set_xlabel('Epoche', fontsize=12)
    axes[0].set_ylabel('Accuracy', fontsize=12)
    axes[0].legend(loc='lower right', fontsize=11, frameon=True)

    # 2. Grafico Loss
    axes[1].plot(epochs_range, loss, label='Training Loss', color='#ff7f0e', linewidth=2.5)
    axes[1].plot(epochs_range, val_loss, label='Validation Loss', color='#d62728', linewidth=2.5)
    axes[1].axvline(x=best_epoch, color='red', linestyle='--', alpha=0.7, label=f'Best Epoch ({best_epoch})')
    axes[1].set_title('Funzione di Costo (Cross-Entropy)', fontsize=14)
    axes[1].set_xlabel('Epoche', fontsize=12)
    axes[1].set_ylabel('Loss', fontsize=12)
    axes[1].legend(loc='upper right', fontsize=11, frameon=True)

    plt.tight_layout()
    plt.savefig('cnn_training_history.svg', format='svg', bbox_inches='tight')
    plt.show()

try:
    plot_training_history(history)
except NameError:
    print("Variabile 'history' non trovata in memoria (Addestramento saltato).")
    print("Caricamento del grafico vettoriale salvato in precedenza...")
    if os.path.exists('cnn_training_history.svg'):
        display(SVG(filename='cnn_training_history.svg'))
    else:
        print("Grafico non trovato. È necessario eseguire il training (Cella 5).")

In [ ]:
"""
Genera la Matrice di Confusione standard affiancata da un istogramma 
degli F1-Score per classe per evidenziare visivamente l'impatto dello sbilanciamento.
Calcola inoltre metriche di validazione probabilistica e relativa (Top-K, Cohen's Kappa, Log-Loss).
"""
from sklearn.metrics import classification_report, confusion_matrix, f1_score, log_loss, cohen_kappa_score, top_k_accuracy_score
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Estrazione delle predizioni sul Validation Set per le metriche finali ...")

y_true = np.concatenate([y for x, y in val_ds], axis=0)
y_pred_probs = cnn_model.predict(val_ds)
y_pred = np.argmax(y_pred_probs, axis=1)

emotions = ['Angry', 'Disgust', 'Fear', 'Happy', 'Neutral', 'Sad', 'Surprise']

kappa = cohen_kappa_score(y_true, y_pred)
loss_val = log_loss(y_true, y_pred_probs)
top2_acc = top_k_accuracy_score(y_true, y_pred_probs, k=2)

print("\n--- METRICHE DI VALIDAZIONE PROBABILISTICA E RELATIVA ---")
print(f"Log-Loss (Cross-Entropy): {loss_val:.4f}")
print(f"Cohen's Kappa (Accordo netto): {kappa:.4f}")
print(f"Top-2 Accuracy (Emozione reale tra le prime 2 previste): {top2_acc:.4f}")

# Report Testuale
report = classification_report(y_true, y_pred, target_names=emotions)
print("\n--- REPORT DI CLASSIFICAZIONE GLOBALE ---")
print(report)

# Calcolo F1-Scores specifici per il grafico
f1_scores = f1_score(y_true, y_pred, average=None)

# Creazione del cruscotto a due riquadri
fig, axes = plt.subplots(1, 2, figsize=(20, 8), gridspec_kw={'width_ratios': [1.5, 1]})
fig.suptitle('Valutazione Finale: CNN Custom (E.R.M.E.S.)', fontsize=18, fontweight='bold', y=1.02)

# Riquadro SINISTRA: Matrice di Confusione
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=emotions, yticklabels=emotions, 
            annot_kws={"size": 12}, ax=axes[0])
axes[0].set_title('Matrice di Confusione', fontsize=15, pad=15)
axes[0].set_ylabel('Classe Reale (True)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Classe Predetta (Predicted)', fontsize=13, fontweight='bold')
axes[0].tick_params(axis='both', which='major', labelsize=11)

# Riquadro DESTRA: Grafico a Barre degli F1-Score
sns.barplot(x=f1_scores, y=emotions, hue=emotions, palette='viridis', ax=axes[1], orient='h', legend=False)
axes[1].set_title('F1-Score per Classe Emotiva', fontsize=15, pad=15)
axes[1].set_xlabel('F1-Score', fontsize=13, fontweight='bold')
axes[1].set_xlim(0, 1.0) # Fissa il limite a 1 (100%)
axes[1].tick_params(axis='both', which='major', labelsize=12)

# Aggiunge i valori numerici sulle barre
for i, v in enumerate(f1_scores):
    axes[1].text(v + 0.02, i, f"{v:.2f}", color='black', va='center', fontweight='bold')

plt.tight_layout()
plt.savefig('cnn_evaluation_dashboard.svg', format='svg', bbox_inches='tight')
plt.show()

In [ ]:
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt
import numpy as np

plt.style.use('seaborn-v0_8-darkgrid')

# Binarizzazione delle label per il calcolo multi-classe (One-vs-Rest)
y_true_bin = label_binarize(y_true, classes=[0, 1, 2, 3, 4, 5, 6])
n_classes = y_true_bin.shape[1]

# Calcolo ROC e AUC per ogni classe
fpr = dict()
tpr = dict()
roc_auc = dict()

for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], np.array(y_pred_probs)[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Calcolo del Macro-Average AUC (Media matematica di tutte le aree)
macro_auc = sum(roc_auc.values()) / n_classes
print(f"--- ANALISI ROC/AUC ---")
print(f"Macro-Average AUC: {macro_auc:.4f} (Capacità globale di separazione delle classi)\n")

# Disegno del grafico
plt.figure(figsize=(10, 8))
colors = ['blue', 'red', 'green', 'orange', 'purple', 'brown', 'pink']

for i, color in zip(range(n_classes), colors):
    plt.plot(fpr[i], tpr[i], color=color, lw=2,
             label=f"ROC {emotions[i]} (AUC = {roc_auc[i]:.2f})")

plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Tasso di Falsi Positivi (FPR)', fontsize=12, fontweight='bold')
plt.ylabel('Tasso di Veri Positivi (TPR)', fontsize=12, fontweight='bold')
plt.title('Curva ROC Multi-classe (E.R.M.E.S. V1)', fontsize=15, pad=15)
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig('roc_curve_ermes.svg', format='svg', bbox_inches='tight')
plt.show()

## Conclusioni della Fase 4

Le evidenze sperimentali derivanti dall'impiego delle Convolutional Neural Networks (CNN) validano l'ipotesi architettonica del progetto E.R.M.E.S., confermando il superamento dei limiti intrinseci dei modelli di Machine Learning classico nel dominio della Computer Vision:

1. **Incremento delle prestazioni e invarianza spaziale:** Si osserva una crescita significativa del Macro F1-Score, passato dal 28% della Baseline al 60%. Tale progressione è riconducibile alla capacità intrinseca delle operazioni di convoluzione di preservare e mappare le relazioni spaziali tra i pixel. A differenza dei modelli basati su vettori appiattiti (flattened), la CNN ha estratto gerarchie di feature in grado di modellare efficacemente la geometria dei tratti somatici.

2. **Ottimizzazione del Cost-Sensitive Learning:** La strategia di gestione dello sbilanciamento delle classi ha prodotto risultati nettamente superiori rispetto alla fase precedente. La CNN ha interiorizzato i pattern visivi caratteristici della complessa classe minoritaria 'Disgust', raggiungendo un F1-Score specifico di 0.58. La robustezza di questo apprendimento al netto del caso fortuito è matematicamente certificata da un punteggio Cohen's Kappa di 0.54 (Accordo Moderato/Buono).

3. **Validazione Probabilistica dell'Ambiguità:** L'Accuracy globale (62%) descrive solo parzialmente la comprensione semantica del modello. L'estrazione delle probabilità continue (attivazioni Softmax) rivela una Top-2 Accuracy vicina all'80%: in 8 casi su 10, l'emozione reale rientra tra le primissime due scelte della rete. Questo dato, affiancato a un Macro-Average AUC eccellente (~0.90), certifica un ottima capacità globale di separazione delle classi. 

4. **Analisi dei limiti strutturali:** I fallimenti predittivi (misurati da una Log-Loss di 1.01) non derivano da errori geometrici grossolani, ma da un'incertezza fisiologica. La distinzione tra emozioni contigue (es. 'Fear' e 'Sad' o 'Fear' e 'Surprise') sconta la sovrapposizione morfologica intrinseca delle espressioni umane, esacerbata dalla bassa risoluzione nativa (48x48 pixel) che rende le micro-contrazioni muscolari difficilmente discriminabili persino per l'occhio umano.

**Prossimi Sviluppi (Fase 5 - Explainable AI):** Il raggiungimento di questo traguardo statistico rappresenta un Upper Bound notevole per un'architettura addestrata ex-novo. Tuttavia, le metriche numeriche non garantiscono che il modello stia "guardando" i punti giusti (potrebbe sfruttare bias del dataset, come filigrane o sfondi). Prima di esplorare il Transfer Learning, è ingegneristicamente imperativo aprire la "scatola nera" della CNN. 
La Fase 5 si concentrerà sull'Interpretabilità (XAI) tramite l'algoritmo Grad-CAM. L'obiettivo è estrarre le mappe dei gradienti spaziali (heatmaps) per dimostrare visivamente che i filtri convoluzionali si attivano in corrispondenza della reale topologia facciale (occhi, bocca, rughe d'espressione), garantendo la totale trasparenza e affidabilità decisionale del sistema.